# Lewis & Short in LatinCy — `lewis_short` component demo

**What this shows:** how the new `lewis_short` spaCy component layers the digitized
**Lewis & Short** dictionary onto a LatinCy pipeline — the offline equivalent of the
lexicon panel in the [Perseus Latin Word Study Tool](https://www.perseus.tufts.edu/hopper/morph?l=ago&la=la#lexicon).

**The key idea — it's a *pure overlay*, not a replacement:**

| Layer | Component | Per token |
|---|---|---|
| morphology + **headword** + **short gloss** | `whitakers_words` | `token._.ww`, `token._.lexicon`, `token._.gloss` |
| full scholarly **dictionary article** | `lewis_short` | `token._.lewis_short` |

`lewis_short` does **not** lemmatize or gloss — it consumes the lemma the pipeline
already assigned and attaches the matching L&S entry. Headword + short def keep coming
from `whitakers_words` (or any upstream lemmatizer). Everything is **local** — no API calls.

> **Note on this demo:** the full LatinCy model (`la_core_web_lg`) isn't required here, so we
> use `spacy.blank("la")` and *simulate* the lemma/POS that LatinCy would assign upstream
> (by setting `token.lemma_` / `token.pos_`). In a real pipeline those come from the tagger
> and lemmatizer, and you simply `nlp.add_pipe("whitakers_words")` then `nlp.add_pipe("lewis_short")`.

## 1. Build the pipeline

In [ ]:
from pathlib import Path
import spacy

PROJECT = Path("..").resolve()
J = PROJECT / "data" / "json"

nlp = spacy.blank("la")
# whitakers_words: headword + short gloss + morphology
nlp.add_pipe("whitakers_words", config={
    "lexicon_path": str(J / "lexicon.json"),
    "analyzer_path": str(J / "analyzer.json"),
})
# lewis_short: the Lewis & Short article overlay (lazy-loads a ~28 MB store)
nlp.add_pipe("lewis_short", config={
    "ls_index_path": str(J / "lewis_short_index.json"),
    "ls_store_path": str(J / "lewis_short.json"),
})
print(nlp.pipe_names)

## 2. A little `study()` helper

Simulates the upstream lemma/POS, runs both components, and pretty-prints what each
layer attached to the token.

In [ ]:
def study(surface, lemma, pos):
    """Build a one-token doc, simulate upstream lemma/POS, run the components."""
    doc = nlp.make_doc(surface)
    tok = doc[0]
    tok.lemma_, tok.pos_ = lemma, pos
    for name in ("whitakers_words", "lewis_short"):
        nlp.get_pipe(name)(doc)
    return doc[0]

def show(tok):
    lex = tok._.lexicon or []
    hw = lex[0]["headword"] if lex else "—"
    gloss = (lex[0].get("glosses") or ["—"])[0] if lex else "—"
    print(f"form        : {tok.text!r}  (pos={tok.pos_}, lemma={tok.lemma_!r})")
    print(f"WW headword : {hw!r}")
    print(f"WW gloss    : {gloss!r}")
    ls = tok._.lewis_short
    if not ls:
        print("L&S overlay : None  — Lewis & Short has no entry for this headword")
        return
    h = ls[0]
    print(f"L&S entry   : orth={h['orth']!r}  pos={h['pos']!r}  parts={h['itype']!r}  (id={h['id']})")
    print(f"L&S ranked  : {[e['orth'] for e in ls]}  ({len(ls)} candidate(s))")
    print(f"handle keys : {sorted(h)}   # note: no 'text' by default")

## 3. A classical word: *ago*

Full stack — WW supplies headword + short gloss; `lewis_short` attaches the L&S article.

In [ ]:
show(study("agit", "ago", "VERB"))

The per-token L&S result is a **lean handle** (metadata only). The heavy entry text
(this one is ~40 KB) is fetched **on demand** by id, so a `Doc` stays light and
serializable:

In [ ]:
ls = nlp.get_pipe("lewis_short")
entry = ls.get_entry(study("agit", "ago", "VERB")._.lewis_short[0]["id"])
print(entry["text"][:400], "...")

## 4. Homograph disambiguation: *malus*

Same surface form, three L&S entries (adj. *mălus* 'bad'; f. *mālus* 'apple-tree'; m. *mālus* 'mast').
The token's POS reranks the candidates — **nothing is dropped**, the best fit floats to the top.

In [ ]:
print("--- as NOUN ---")
show(study("malus", "malus", "NOUN"))
print()
print("--- as ADJ ---")
show(study("malus", "malus", "ADJ"))

## 5. Orthographic recovery: assimilation

WW stores the etymological un-assimilated spelling (*adcedo* = ad + cedo); L&S indexes the
classical assimilated form (*accedo*). When a direct lookup misses, `lewis_short` retries
with the assimilated spelling — so *adcedo* still finds *ac-cēdo*.

In [ ]:
show(study("adcedit", "adcedo", "VERB"))

## 6. The honest gap: a medieval word *abbatia*

*abbatia* ('abbey') is in Whitaker's Words but **not** in classical Lewis & Short.
The user still gets a full WW gloss; the L&S overlay is simply `None` — which is the
correct signal, not an error. A UI renders the L&S panel only when it's populated.

In [ ]:
show(study("abbatia", "abbatia", "NOUN"))

## 7. Opt-in: inline the full text

For batch/export use you can ask the component to inline the full article on every token
with `include_text=True` (heavier `Doc`, no per-id fetch needed). The default stays lean.

In [ ]:
nlp_full = spacy.blank("la")
nlp_full.add_pipe("lewis_short", config={
    "ls_index_path": str(J / "lewis_short_index.json"),
    "ls_store_path": str(J / "lewis_short.json"),
    "include_text": True,
})
doc = nlp_full.make_doc("agit")
doc[0].lemma_, doc[0].pos_ = "ago", "VERB"
nlp_full.get_pipe("lewis_short")(doc)
h = doc[0]._.lewis_short[0]
print("handle keys now include text:", sorted(h))
print("text length:", len(h["text"]))

## 8. Coverage reality check

How much of the WW lexicon does the L&S overlay reach? This is *headword* coverage of the
overlay, **not** what a reader experiences on real text (the unmatched tail is overwhelmingly
rare/medieval/neo-Latin words that seldom appear, and they still get a WW gloss).

In [ ]:
import json
from latincy_lexicon.align.lewis_short import align_lexicon_to_ls

idx = json.load(open(J / "lewis_short_index.json"))
store = json.load(open(J / "lewis_short.json"))
lex = json.load(open(J / "lexicon.json"))
_, stats = align_lexicon_to_ls(lex, idx, store)

tot, al = stats["entries_total"], stats["entries_aligned"]
print(f"lexicon entries     : {tot:,}")
print(f"aligned to L&S      : {al:,}  ({al/tot:.1%})")
print(f"  via assimilation  : {stats['entries_assimilated']:,}")
print(f"  homograph clusters: {stats['entries_disambiguated']:,}")
print(f"unmatched (WW-only) : {stats['entries_unmatched']:,}  — medieval / late / neo-Latin / rare")

## Where we landed

- **`lewis_short` is a pure, optional overlay.** Drop it after `whitakers_words` and every
  recognized token gains its Lewis & Short article when one exists.
- **Headword + short gloss** come from `whitakers_words`; **the full article** from `lewis_short`.
  Together they reproduce the Perseus Word-Study-Tool experience — fully offline.
- **Lean by default**, `include_text=True` or `get_entry(id)` for the full text.
- **Homographs ranked** by POS (never dropped); **assimilated spellings recovered** automatically.
- **~71% headword coverage** is near the structural ceiling for L&S alone; the gap is
  vocabulary L&S never contained, and those tokens still get a WW gloss.

_Next: a TEI→HTML renderer in `latincy-lexicon-site` to display the L&S `text` faithfully
(sense tree + citations) instead of the flattened string shown here._